# Train on Kaggle
Upload `kaggle_upload/spacenet_paris.zip` as a Kaggle Dataset, add it to this notebook, and turn on a GPU (Settings -> Accelerator). Trains U-Net, DeepLabV3+ and SegFormer one after another; checkpoints and logs go to `/kaggle/working/`. Internet must be on (pretrained ImageNet weights are downloaded).

In [ ]:
import glob, os, tarfile

# 1. Unpack the dataset from /kaggle/input/ to /kaggle/working/spacenet
WORK = '/kaggle/working/spacenet'
tars = glob.glob('/kaggle/input/**/spacenet_data.bin', recursive=True)
assert tars, 'spacenet_data.bin not found under /kaggle/input/'
os.makedirs(WORK, exist_ok=True)
with tarfile.open(tars[0]) as t:  # .bin so Kaggle does not auto-extract it; tarfile ignores the extension
    t.extractall(WORK)            # yields data/images, data/masks, data/splits.csv and src/
assert os.path.exists(f'{WORK}/src/train.py'), 'src/train.py missing after extraction'
n_images = len(glob.glob(f'{WORK}/data/images/*.png'))
print(os.listdir(WORK), '| images found:', n_images)
assert n_images == 617, f'expected 617 images, found {n_images}'


In [ ]:
# 2. Install segmentation-models-pytorch if missing
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
    import segmentation_models_pytorch as smp
import albumentations
print('smp', smp.__version__, '| albumentations', albumentations.__version__)

In [ ]:
# 3. Check the GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: no GPU - turn one on in Settings -> Accelerator before training')

In [ ]:
# 4. Train the 3 models one after another (num_workers=2 on Kaggle)
for model in ['unet', 'deeplabv3plus', 'segformer']:
    print('=' * 20, model, '=' * 20, flush=True)
    subprocess.run([sys.executable, f'{WORK}/src/train.py', '--model', model,
                    '--data_dir', f'{WORK}/data', '--out_dir', '/kaggle/working',
                    '--num_workers', '2'], check=True, cwd=f'{WORK}/src')

In [ ]:
# 5. Results
import pandas as pd
for model in ['unet', 'deeplabv3plus', 'segformer']:
    log = pd.read_csv(f'/kaggle/working/results/{model}_log.csv')
    print(model, '| epochs run:', len(log), '| best val mIoU:', log['miou'].max())
print(os.listdir('/kaggle/working/models'), os.listdir('/kaggle/working/results'))